# Chapter 4: A Deeper Dive into Loading Data
**Module 01 – Deep Learning with PyTorch: Introduction**

> *Instructor: Maham Faisal Khan, Senior Data Scientist*

## 4.1 Datasets and DataLoaders

PyTorch provides two core classes for handling data:

| Class | Purpose |
|---|---|
| `Dataset` | Stores samples and their labels |
| `DataLoader` | Wraps a Dataset; provides batching, shuffling, and parallel loading |

Working with a custom dataset requires implementing:
- `__init__()` — initialize file paths, transforms
- `__len__()` — return the total number of samples
- `__getitem__(idx)` — return the sample at index `idx`

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# Example: Loading animals dataset
# pd.read_csv('animals.csv')  # Uncomment with actual data

# Simulated animal dataset structure
data_dict = {
    'animal_name': ['skimmer', 'gull', 'seahorse', 'tuatara'],
    'hair':    [0, 0, 0, 0],
    'feathers':[1, 1, 0, 0],
    'eggs':    [1, 1, 1, 1],
    'milk':    [0, 0, 0, 0],
    'type':    [2, 2, 4, 3]
}
df = pd.DataFrame(data_dict)
print(df)

## 4.2 Creating a Custom PyTorch Dataset

In [ ]:
class AnimalDataset(Dataset):
    def __init__(self, dataframe):
        # Features: all numeric columns except type
        self.X = torch.tensor(
            dataframe.drop(['animal_name', 'type'], axis=1).values,
            dtype=torch.float32
        )
        # Labels: type column
        self.y = torch.tensor(dataframe['type'].values, dtype=torch.long)

    def __len__(self):
        """Return the total number of samples."""
        return len(self.y)

    def __getitem__(self, idx):
        """Return a single (features, label) pair."""
        return self.X[idx], self.y[idx]


# Instantiate dataset
dataset = AnimalDataset(df)
print(f"Dataset size: {len(dataset)}")
print(f"Sample 0: {dataset[0]}")

## 4.3 Creating DataLoaders

In [ ]:
# Create DataLoader with batch size and shuffling
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# Iterate through batches
for batch_idx, (features, labels) in enumerate(dataloader):
    print(f"Batch {batch_idx}: features shape={features.shape}, labels={labels}")

print("\nDataLoader iterated successfully!")

## 4.4 Full Training Loop with Custom Dataset

In [ ]:
import torch.nn as nn

# Define a simple model
model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 5)  # 5 animal classes
)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(5):
    model.train()
    total_loss = 0
    for features, labels in dataloader:
        # Forward pass
        outputs = model(features)
        loss = criterion(outputs, labels)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/5 | Loss: {total_loss:.4f}")

## Summary

| Component | Role |
|---|---|
| `Dataset.__len__()` | Reports dataset size |
| `Dataset.__getitem__()` | Returns one sample |
| `DataLoader` | Handles batching, shuffling, parallel loading |
| `optimizer.zero_grad()` | Clear gradients before each backward pass |
| `loss.backward()` | Compute gradients |
| `optimizer.step()` | Update weights |